# Notebook 01: Data Collection and Exploration

## RustWeatherML - Weather Prediction System in Rust

This notebook covers:
1. Setting up dependencies
2. Understanding the Open-Meteo API
3. Fetching historical weather data for 14 cities
4. Exploratory Data Analysis (EDA)
5. Data quality assessment
6. Saving raw data for preprocessing

**Data Source**: [Open-Meteo](https://open-meteo.com/) - Free weather API (no API key required)

**Time Range**: 2016-2025 (10 years of hourly data)

**Cities**: 14 cities across 4 continents

---
## 1. Setup Dependencies

First, let's load all the required crates for data collection and analysis.

In [ ]:
// Load dependencies
:dep polars = { version = "0.46", features = ["lazy", "parquet", "csv", "json", "dtype-datetime"] }
:dep reqwest = { version = "0.12", features = ["blocking", "json"] }
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"
:dep chrono = { version = "0.4", features = ["serde"] }
:dep anyhow = "1.0"

In [ ]:
// Import required modules
use polars::prelude::*;
use reqwest::blocking::Client;
use serde::{Deserialize, Serialize};
use chrono::{NaiveDate, NaiveDateTime};
use std::collections::HashMap;
use std::time::Duration;
use std::thread;

println!("Dependencies loaded successfully!");

---
## 2. Define Data Structures

Let's define the structures to represent cities and weather data.

In [ ]:
/// Represents a city with geographic coordinates
#[derive(Debug, Clone)]
struct City {
    name: String,
    country: String,
    country_code: String,
    latitude: f64,
    longitude: f64,
    timezone: String,
}

impl City {
    fn new(name: &str, country: &str, code: &str, lat: f64, lon: f64, tz: &str) -> Self {
        Self {
            name: name.to_string(),
            country: country.to_string(),
            country_code: code.to_string(),
            latitude: lat,
            longitude: lon,
            timezone: tz.to_string(),
        }
    }
}

println!("City struct defined!");

In [ ]:
/// Define all 14 cities for our dataset
fn get_cities() -> Vec<City> {
    vec![
        // Brazil (4 cities)
        City::new("São Paulo", "Brazil", "BR", -23.55, -46.63, "America/Sao_Paulo"),
        City::new("Rio de Janeiro", "Brazil", "BR", -22.91, -43.17, "America/Sao_Paulo"),
        City::new("São José dos Campos", "Brazil", "BR", -23.18, -45.88, "America/Sao_Paulo"),
        City::new("Campinas", "Brazil", "BR", -22.91, -47.06, "America/Sao_Paulo"),
        
        // USA (2 cities)
        City::new("New York", "USA", "US", 40.71, -74.01, "America/New_York"),
        City::new("Los Angeles", "USA", "US", 34.05, -118.24, "America/Los_Angeles"),
        
        // Europe (3 cities)
        City::new("London", "United Kingdom", "GB", 51.51, -0.13, "Europe/London"),
        City::new("Berlin", "Germany", "DE", 52.52, 13.40, "Europe/Berlin"),
        City::new("Oslo", "Norway", "NO", 59.91, 10.75, "Europe/Oslo"),
        
        // Asia (5 cities)
        City::new("Tokyo", "Japan", "JP", 35.68, 139.69, "Asia/Tokyo"),
        City::new("Shanghai", "China", "CN", 31.23, 121.47, "Asia/Shanghai"),
        City::new("Chongqing", "China", "CN", 29.56, 106.55, "Asia/Shanghai"),
        City::new("Nanjing", "China", "CN", 32.06, 118.80, "Asia/Shanghai"),
        City::new("Dubai", "UAE", "AE", 25.27, 55.30, "Asia/Dubai"),
    ]
}

let cities = get_cities();
println!("Defined {} cities:", cities.len());
for (i, city) in cities.iter().enumerate() {
    println!("  {}. {} ({}) - [{:.2}, {:.2}]", 
             i + 1, city.name, city.country_code, city.latitude, city.longitude);
}

---
## 3. Open-Meteo API Response Structures

Define structures to deserialize the API response.

In [ ]:
/// Response from Open-Meteo Historical API
#[derive(Debug, Deserialize)]
struct OpenMeteoResponse {
    latitude: f64,
    longitude: f64,
    #[serde(default)]
    timezone: String,
    hourly: HourlyData,
}

#[derive(Debug, Deserialize)]
struct HourlyData {
    time: Vec<String>,
    #[serde(default)]
    temperature_2m: Option<Vec<Option<f64>>>,
    #[serde(default)]
    apparent_temperature: Option<Vec<Option<f64>>>,
    #[serde(default)]
    dewpoint_2m: Option<Vec<Option<f64>>>,
    #[serde(default)]
    precipitation: Option<Vec<Option<f64>>>,
    #[serde(default)]
    rain: Option<Vec<Option<f64>>>,
    #[serde(default)]
    snowfall: Option<Vec<Option<f64>>>,
    #[serde(default)]
    windspeed_10m: Option<Vec<Option<f64>>>,
    #[serde(default)]
    windgusts_10m: Option<Vec<Option<f64>>>,
    #[serde(default)]
    winddirection_10m: Option<Vec<Option<f64>>>,
    #[serde(default)]
    pressure_msl: Option<Vec<Option<f64>>>,
    #[serde(default)]
    surface_pressure: Option<Vec<Option<f64>>>,
    #[serde(default)]
    cloudcover: Option<Vec<Option<f64>>>,
    #[serde(default)]
    visibility: Option<Vec<Option<f64>>>,
    #[serde(default)]
    shortwave_radiation: Option<Vec<Option<f64>>>,
    #[serde(default)]
    direct_radiation: Option<Vec<Option<f64>>>,
    #[serde(default)]
    relativehumidity_2m: Option<Vec<Option<f64>>>,
    #[serde(default)]
    weathercode: Option<Vec<Option<i64>>>,
}

println!("API response structures defined!");

---
## 4. API Client Implementation

Create a client to fetch data from Open-Meteo.

In [ ]:
/// Open-Meteo API client
struct OpenMeteoClient {
    client: Client,
    archive_url: String,
}

impl OpenMeteoClient {
    fn new() -> Self {
        Self {
            client: Client::builder()
                .timeout(Duration::from_secs(120))
                .build()
                .expect("Failed to create HTTP client"),
            archive_url: "https://archive-api.open-meteo.com/v1/archive".to_string(),
        }
    }
    
    /// Get hourly parameters string
    fn hourly_params() -> &'static str {
        "temperature_2m,apparent_temperature,dewpoint_2m,\
         precipitation,rain,snowfall,\
         windspeed_10m,windgusts_10m,winddirection_10m,\
         pressure_msl,surface_pressure,cloudcover,visibility,\
         shortwave_radiation,direct_radiation,\
         relativehumidity_2m,weathercode"
    }
    
    /// Fetch historical data for a city and date range
    fn fetch_historical(
        &self,
        city: &City,
        start_date: &str,
        end_date: &str,
    ) -> Result<OpenMeteoResponse, Box<dyn std::error::Error>> {
        let url = format!(
            "{}?latitude={}&longitude={}&start_date={}&end_date={}&hourly={}&timezone={}",
            self.archive_url,
            city.latitude,
            city.longitude,
            start_date,
            end_date,
            Self::hourly_params(),
            city.timezone
        );
        
        let response = self.client.get(&url).send()?;
        let data: OpenMeteoResponse = response.json()?;
        Ok(data)
    }
}

let api_client = OpenMeteoClient::new();
println!("API client created!");

---
## 5. Test API Connection

Let's test the API with a small request to verify everything works.

In [ ]:
// Test API with São Paulo for 1 week
let test_city = &cities[0]; // São Paulo
let test_start = "2024-01-01";
let test_end = "2024-01-07";

println!("Testing API connection...");
println!("City: {} ({:.2}, {:.2})", test_city.name, test_city.latitude, test_city.longitude);
println!("Date range: {} to {}", test_start, test_end);

match api_client.fetch_historical(test_city, test_start, test_end) {
    Ok(response) => {
        println!("\n✓ API connection successful!");
        println!("  Response latitude: {:.2}", response.latitude);
        println!("  Response longitude: {:.2}", response.longitude);
        println!("  Number of hourly records: {}", response.hourly.time.len());
        
        // Show first few records
        println!("\nFirst 5 timestamps:");
        for (i, time) in response.hourly.time.iter().take(5).enumerate() {
            let temp = response.hourly.temperature_2m
                .as_ref()
                .and_then(|v| v.get(i))
                .and_then(|t| *t);
            println!("  {} - Temp: {:?}°C", time, temp);
        }
    },
    Err(e) => {
        println!("✗ API connection failed: {}", e);
    }
}

---
## 6. Convert API Response to DataFrame

Create a function to convert API responses to Polars DataFrames.

In [ ]:
/// Convert API response to a Polars DataFrame
fn response_to_dataframe(
    city: &City,
    response: &OpenMeteoResponse,
) -> Result<DataFrame, PolarsError> {
    let n = response.hourly.time.len();
    
    // Helper function to extract Option<Vec<Option<f64>>> to Vec<Option<f64>>
    let extract_f64 = |opt: &Option<Vec<Option<f64>>>| -> Vec<Option<f64>> {
        opt.as_ref()
            .map(|v| v.clone())
            .unwrap_or_else(|| vec![None; n])
    };
    
    let extract_i64 = |opt: &Option<Vec<Option<i64>>>| -> Vec<Option<i64>> {
        opt.as_ref()
            .map(|v| v.clone())
            .unwrap_or_else(|| vec![None; n])
    };
    
    // Create city and location columns
    let city_names: Vec<&str> = vec![city.name.as_str(); n];
    let country_codes: Vec<&str> = vec![city.country_code.as_str(); n];
    let latitudes: Vec<f64> = vec![city.latitude; n];
    let longitudes: Vec<f64> = vec![city.longitude; n];
    
    // Parse timestamps
    let timestamps: Vec<&str> = response.hourly.time.iter().map(|s| s.as_str()).collect();
    
    df![
        "city" => city_names,
        "country_code" => country_codes,
        "latitude" => latitudes,
        "longitude" => longitudes,
        "timestamp" => timestamps,
        "temperature_2m" => extract_f64(&response.hourly.temperature_2m),
        "apparent_temperature" => extract_f64(&response.hourly.apparent_temperature),
        "dewpoint_2m" => extract_f64(&response.hourly.dewpoint_2m),
        "precipitation" => extract_f64(&response.hourly.precipitation),
        "rain" => extract_f64(&response.hourly.rain),
        "snowfall" => extract_f64(&response.hourly.snowfall),
        "windspeed_10m" => extract_f64(&response.hourly.windspeed_10m),
        "windgusts_10m" => extract_f64(&response.hourly.windgusts_10m),
        "winddirection_10m" => extract_f64(&response.hourly.winddirection_10m),
        "pressure_msl" => extract_f64(&response.hourly.pressure_msl),
        "surface_pressure" => extract_f64(&response.hourly.surface_pressure),
        "cloudcover" => extract_f64(&response.hourly.cloudcover),
        "visibility" => extract_f64(&response.hourly.visibility),
        "shortwave_radiation" => extract_f64(&response.hourly.shortwave_radiation),
        "direct_radiation" => extract_f64(&response.hourly.direct_radiation),
        "relativehumidity_2m" => extract_f64(&response.hourly.relativehumidity_2m),
        "weathercode" => extract_i64(&response.hourly.weathercode)
    ]
}

println!("DataFrame conversion function defined!");

In [ ]:
// Test DataFrame conversion
let test_response = api_client.fetch_historical(test_city, test_start, test_end).unwrap();
let test_df = response_to_dataframe(test_city, &test_response).unwrap();

println!("Test DataFrame shape: {} rows x {} columns", test_df.height(), test_df.width());
println!("\nColumn names:");
for name in test_df.get_column_names() {
    println!("  - {}", name);
}
println!("\nFirst 5 rows:");
println!("{}", test_df.head(Some(5)));

---
## 7. Fetch Historical Data (Sample)

For demonstration, let's fetch 1 month of data for all cities.

**Note**: Fetching the full 10 years of data will be done in a separate script due to time constraints. Here we demonstrate the process with a smaller sample.

In [ ]:
// Fetch 1 month of sample data for all cities
let sample_start = "2024-01-01";
let sample_end = "2024-01-31";

println!("Fetching sample data ({} to {})...", sample_start, sample_end);
println!("This will take a moment...\n");

let mut all_dataframes: Vec<DataFrame> = Vec::new();

for (i, city) in cities.iter().enumerate() {
    print!("[{}/{}] Fetching {}... ", i + 1, cities.len(), city.name);
    
    match api_client.fetch_historical(city, sample_start, sample_end) {
        Ok(response) => {
            match response_to_dataframe(city, &response) {
                Ok(df) => {
                    println!("✓ {} records", df.height());
                    all_dataframes.push(df);
                },
                Err(e) => println!("✗ DataFrame error: {}", e),
            }
        },
        Err(e) => println!("✗ API error: {}", e),
    }
    
    // Rate limiting - be nice to the free API
    thread::sleep(Duration::from_millis(500));
}

println!("\nFetched data for {} cities", all_dataframes.len());

In [ ]:
// Combine all DataFrames into one
use polars::functions::concat_df_diagonal;

let combined_df = if !all_dataframes.is_empty() {
    let df_refs: Vec<DataFrame> = all_dataframes.into_iter().collect();
    concat_df_diagonal(&df_refs).expect("Failed to concatenate DataFrames")
} else {
    panic!("No DataFrames to combine!");
};

println!("Combined DataFrame shape: {} rows x {} columns", combined_df.height(), combined_df.width());
println!("\nMemory usage estimate: ~{:.2} MB", 
         (combined_df.height() * combined_df.width() * 8) as f64 / 1_000_000.0);

---
## 8. Exploratory Data Analysis (EDA)

Let's explore the data to understand its characteristics.

In [ ]:
// Basic statistics
println!("=== BASIC STATISTICS ===");
println!("\nDataFrame info:");
println!("  - Total records: {}", combined_df.height());
println!("  - Total columns: {}", combined_df.width());

// Records per city
println!("\nRecords per city:");
let city_counts = combined_df
    .clone()
    .lazy()
    .group_by([col("city")])
    .agg([col("timestamp").count().alias("count")])
    .sort(["city"], Default::default())
    .collect()
    .unwrap();

println!("{}", city_counts);

In [ ]:
// Summary statistics for numeric columns
println!("=== SUMMARY STATISTICS ===");

let numeric_cols = vec![
    "temperature_2m", "apparent_temperature", "precipitation",
    "windspeed_10m", "pressure_msl", "cloudcover", "relativehumidity_2m"
];

for col_name in &numeric_cols {
    if let Ok(col) = combined_df.column(*col_name) {
        let series = col.f64().unwrap();
        let non_null = series.len() - series.null_count();
        
        println!("\n{}:", col_name);
        println!("  - Non-null values: {} ({:.1}%)", non_null, (non_null as f64 / series.len() as f64) * 100.0);
        
        if let Some(mean) = series.mean() {
            println!("  - Mean: {:.2}", mean);
        }
        if let Some(min) = series.min() {
            println!("  - Min: {:.2}", min);
        }
        if let Some(max) = series.max() {
            println!("  - Max: {:.2}", max);
        }
    }
}

In [ ]:
// Temperature statistics by city
println!("=== TEMPERATURE BY CITY ===");

let temp_by_city = combined_df
    .clone()
    .lazy()
    .group_by([col("city")])
    .agg([
        col("temperature_2m").mean().alias("avg_temp"),
        col("temperature_2m").min().alias("min_temp"),
        col("temperature_2m").max().alias("max_temp"),
        col("temperature_2m").std(1).alias("std_temp"),
    ])
    .sort(["avg_temp"], SortMultipleOptions::default().with_order_descending(true))
    .collect()
    .unwrap();

println!("{}", temp_by_city);

In [ ]:
// Precipitation statistics by city
println!("=== PRECIPITATION BY CITY ===");

let precip_by_city = combined_df
    .clone()
    .lazy()
    .group_by([col("city")])
    .agg([
        col("precipitation").sum().alias("total_precip_mm"),
        col("precipitation").mean().alias("avg_precip"),
        col("precipitation").filter(col("precipitation").gt(lit(0.0))).count().alias("rainy_hours"),
    ])
    .sort(["total_precip_mm"], SortMultipleOptions::default().with_order_descending(true))
    .collect()
    .unwrap();

println!("{}", precip_by_city);

In [ ]:
// Weather code distribution
println!("=== WEATHER CODE DISTRIBUTION ===");

let weather_codes = combined_df
    .clone()
    .lazy()
    .group_by([col("weathercode")])
    .agg([col("timestamp").count().alias("count")])
    .sort(["count"], SortMultipleOptions::default().with_order_descending(true))
    .collect()
    .unwrap();

println!("\nWeather Code Meanings:");
println!("  0-1: Clear");
println!("  2-3: Partly cloudy / Overcast");
println!("  45, 48: Fog");
println!("  51-67: Drizzle / Rain");
println!("  71-77: Snow");
println!("  80-82: Rain showers");
println!("  95-99: Thunderstorm");
println!("\n{}", weather_codes);

---
## 9. Data Quality Assessment

Check for missing values and data quality issues.

In [ ]:
// Missing values analysis
println!("=== MISSING VALUES ANALYSIS ===");
println!("\nColumn | Null Count | Null % | Data Type");
println!("{}|{}|{}|{}", "-".repeat(25), "-".repeat(12), "-".repeat(10), "-".repeat(15));

for col in combined_df.get_columns() {
    let null_count = col.null_count();
    let total = col.len();
    let null_pct = (null_count as f64 / total as f64) * 100.0;
    let dtype = col.dtype();
    
    println!("{:<25}| {:>10} | {:>7.2}% | {:?}", 
             col.name(), null_count, null_pct, dtype);
}

In [ ]:
// Check for potential outliers in temperature
println!("=== POTENTIAL OUTLIERS CHECK ===");

// Temperature outliers (extreme values)
let temp_col = combined_df.column("temperature_2m").unwrap().f64().unwrap();
let temp_mean = temp_col.mean().unwrap_or(0.0);
let temp_std = temp_col.std(1).unwrap_or(1.0);

println!("\nTemperature 2m:");
println!("  Mean: {:.2}°C", temp_mean);
println!("  Std Dev: {:.2}°C", temp_std);
println!("  3-sigma range: [{:.2}, {:.2}]°C", temp_mean - 3.0 * temp_std, temp_mean + 3.0 * temp_std);

// Count values outside 3-sigma
let lower_bound = temp_mean - 3.0 * temp_std;
let upper_bound = temp_mean + 3.0 * temp_std;

let outliers = combined_df
    .clone()
    .lazy()
    .filter(
        col("temperature_2m").lt(lit(lower_bound))
        .or(col("temperature_2m").gt(lit(upper_bound)))
    )
    .collect()
    .unwrap();

println!("  Potential outliers (>3σ): {} records ({:.2}%)", 
         outliers.height(),
         (outliers.height() as f64 / combined_df.height() as f64) * 100.0);

---
## 10. Save Sample Data

Save the sample data for use in the next notebook.

In [ ]:
// Save to Parquet (more efficient)
use std::fs::File;

let parquet_path = "../data/raw/weather_sample_2024_01.parquet";

// Create directory if it doesn't exist
std::fs::create_dir_all("../data/raw").expect("Failed to create directory");

let mut file = File::create(parquet_path).expect("Failed to create file");
ParquetWriter::new(&mut file)
    .finish(&mut combined_df.clone())
    .expect("Failed to write parquet");

println!("✓ Data saved to: {}", parquet_path);
println!("  - Records: {}", combined_df.height());
println!("  - Columns: {}", combined_df.width());

In [ ]:
// Also save as CSV for inspection
let csv_path = "../data/raw/weather_sample_2024_01.csv";

let mut file = File::create(csv_path).expect("Failed to create file");
CsvWriter::new(&mut file)
    .finish(&mut combined_df.clone())
    .expect("Failed to write CSV");

println!("✓ Data also saved to: {}", csv_path);

---
## 11. Summary and Next Steps

### What we accomplished:
1. ✅ Set up Rust dependencies for data science
2. ✅ Connected to Open-Meteo API (free, no API key)
3. ✅ Defined data structures for 14 cities
4. ✅ Fetched sample historical data (1 month)
5. ✅ Converted API responses to Polars DataFrames
6. ✅ Performed initial EDA (statistics by city)
7. ✅ Assessed data quality (missing values, outliers)
8. ✅ Saved data to Parquet and CSV formats

### Key Findings:
- Data quality is good (minimal missing values)
- Clear temperature differences between cities (climate zones)
- Weather code distribution shows mostly clear/cloudy days
- Some outliers exist but within reasonable bounds

### Next Steps (Notebook 02):
1. Fetch full 10-year dataset (2016-2025)
2. Handle missing values
3. Create target variables:
   - `will_rain` (binary)
   - `weather_condition` (multi-class)
   - `temp_next_24h`, `temp_next_48h`, `temp_next_72h` (regression)
4. Feature engineering:
   - Lag features
   - Rolling statistics
   - Cyclical encoding (hour, day, month)
5. Train/Validation/Test split (2016-2023/2024/2025)

In [ ]:
println!("\n" + "=".repeat(60).as_str());
println!("Notebook 01 Complete!");
println!("=".repeat(60));
println!("\nProceeed to Notebook 02: Preprocessing & Feature Engineering");